# 📊 B2B Sales Pipeline - Exploratory Data Analysis

---

## 📌 Project Overview

This analysis examines B2B sales pipeline data from a computer hardware company, covering:
- **85 Accounts** (Customer companies)
- **7 Products** (Hardware products)
- **35 Sales Agents** across multiple regional offices
- **8,800 Sales Opportunities** over 14 months (Oct 2016 - Dec 2017)

---

## 🎯 Analysis Questions

1. **How is each sales team performing compared to the rest?**
2. **Are any sales agents lagging behind?**
3. **Are there any quarter-over-quarter trends?**
4. **Do any products have better win rates?**

---

## 📦 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set Plotly default theme
import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 📂 Load Data

In [2]:
# Load all datasets
accounts = pd.read_csv('cleaned_accounts.csv')
products = pd.read_csv('cleaned_products.csv')
sales_teams = pd.read_csv('cleaned_sales_teams.csv')
sales_pipeline = pd.read_csv('cleaned_sales_pipeline.csv')

print(f"✓ Accounts: {len(accounts):,} records")
print(f"✓ Products: {len(products):,} records")
print(f"✓ Sales Teams: {len(sales_teams):,} records")
print(f"✓ Sales Pipeline: {len(sales_pipeline):,} records")

✓ Accounts: 85 records
✓ Products: 7 records
✓ Sales Teams: 35 records
✓ Sales Pipeline: 8,800 records


## 🔧 Data Preparation

In [3]:
# Convert date columns
sales_pipeline['engage_date'] = pd.to_datetime(sales_pipeline['engage_date'], errors='coerce')
sales_pipeline['close_date'] = pd.to_datetime(sales_pipeline['close_date'], errors='coerce')

# Fill missing close_value with 0
sales_pipeline['close_value'] = sales_pipeline['close_value'].fillna(0)

print("\n📋 Sales Pipeline Overview:")
print(f"Shape: {sales_pipeline.shape}")
print("\nDeal Stage Distribution:")
print(sales_pipeline['deal_stage'].value_counts())
print(f"\nTotal Revenue (Won Deals): ${sales_pipeline[sales_pipeline['deal_stage'] == 'Won']['close_value'].sum():,.0f}")


📋 Sales Pipeline Overview:
Shape: (8800, 13)

Deal Stage Distribution:
deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

Total Revenue (Won Deals): $10,005,534


## 📊 Overall Pipeline Overview

In [4]:
# Deal Stage Distribution
deal_stage_counts = sales_pipeline['deal_stage'].value_counts()

fig = go.Figure(data=[go.Pie(
    labels=deal_stage_counts.index,
    values=deal_stage_counts.values,
    hole=0.4,
    marker=dict(colors=['#2ecc71', '#e74c3c', '#f39c12', '#3498db']),
    textinfo='label+percent',
    textfont_size=14
)])

fig.update_layout(
    title=dict(
        text='<b>Deal Stage Distribution</b>',
        font=dict(size=20)
    ),
    height=500,
    showlegend=True
)

fig.show()

### 📝 Key Insights:

- **Won Deals**: 48.2% of all opportunities (4,238 deals)
- **Lost Deals**: 28.1% (2,473 deals)
- **Active Pipeline**: 23.8% still in Engaging/Prospecting stages
- **Overall Win Rate**: 63.2% (Won / (Won + Lost))

---

# 🏆 SECTION 1: Sales Team Performance Analysis

---

In [6]:
# Aggregate by sales agent
team_performance = sales_pipeline.groupby('sales_agent').agg({
    'opportunity_id': 'count',
    'close_value': 'sum'
}).rename(columns={'opportunity_id': 'total_opportunities', 'close_value': 'total_revenue'})

# Won/Lost deals
won_deals = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby('sales_agent').size()
lost_deals = sales_pipeline[sales_pipeline['deal_stage'] == 'Lost'].groupby('sales_agent').size()

team_performance['won_deals'] = won_deals
team_performance['lost_deals'] = lost_deals
team_performance = team_performance.fillna(0)

# Win rate
team_performance['completed_deals'] = team_performance['won_deals'] + team_performance['lost_deals']
team_performance['win_rate_percent'] = (team_performance['won_deals'] / team_performance['completed_deals'] * 100).round(2)

# Average deal value
avg_deal_value = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby('sales_agent')['close_value'].mean()
team_performance['avg_deal_value'] = avg_deal_value.round(0)

# Merge with team info
team_performance = team_performance.merge(
    sales_teams[['sales_agent', 'manager', 'regional_office']], 
    left_index=True, 
    right_on='sales_agent',
    how='left'
)

# Team comparison
team_performance['team_avg_revenue'] = team_performance.groupby('manager')['total_revenue'].transform('mean')
team_performance['pct_of_team_avg'] = (team_performance['total_revenue'] / team_performance['team_avg_revenue'] * 100).round(1)

print("✅ Team performance data prepared")

✅ Team performance data prepared


## 📊 Plot 1.1: Top 15 Sales Agents by Revenue

In [7]:
# Top 15 agents by revenue
top_15 = team_performance.nlargest(15, 'total_revenue').sort_values('total_revenue')

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_15['sales_agent'],
    x=top_15['total_revenue'],
    orientation='h',
    marker=dict(
        color=top_15['total_revenue'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title="Revenue ($)")
    ),
    text=[f'${v:,.0f}' for v in top_15['total_revenue']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Revenue: $%{x:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Top 15 Sales Agents by Total Revenue</b>',
        font=dict(size=20)
    ),
    xaxis_title='Total Revenue ($)',
    yaxis_title='Sales Agent',
    height=600,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **Top Performer**: Darcel Schlecht generated $1.15M in revenue (256% of team average)
- **Top 5 agents** account for ~30% of total company revenue
- Significant revenue disparity between top and median performers
- Performance is concentrated in a small group of high achievers

## 📊 Plot 1.2: Win Rate Distribution Across Sales Agents

In [8]:
# Win rate distribution
win_rates = team_performance['win_rate_percent'].dropna()
mean_win_rate = win_rates.mean()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=win_rates,
    nbinsx=25,
    marker=dict(color='#3498db', line=dict(color='white', width=1)),
    hovertemplate='Win Rate: %{x:.1f}%<br>Count: %{y}<extra></extra>'
))

fig.add_vline(
    x=mean_win_rate,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {mean_win_rate:.1f}%",
    annotation_position="top"
)

fig.update_layout(
    title=dict(
        text='<b>Win Rate Distribution Across Sales Agents</b>',
        font=dict(size=20)
    ),
    xaxis_title='Win Rate (%)',
    yaxis_title='Number of Agents',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **Average Win Rate**: 62.7% across all agents
- Distribution is relatively **normal** with slight left skew
- Most agents cluster between 58-66% win rate
- Few outliers on both high and low ends indicate consistent training/processes

## 📊 Plot 1.3: Total Revenue by Regional Office

In [9]:
# Revenue by regional office
office_revenue = team_performance.groupby('regional_office')['total_revenue'].sum().sort_values(ascending=False)
office_count = team_performance.groupby('regional_office').size()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=office_revenue.index,
    y=office_revenue.values,
    marker=dict(
        color=['#2ecc71', '#3498db', '#e74c3c'],
        line=dict(color='white', width=2)
    ),
    text=[f'${v/1e6:.2f}M<br>({office_count[k]} agents)' for k, v in office_revenue.items()],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Total Revenue by Regional Office</b>',
        font=dict(size=20)
    ),
    xaxis_title='Regional Office',
    yaxis_title='Total Revenue ($)',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **East Office** leads with highest total revenue despite fewer agents
- **Central Office** has most agents but middle-tier revenue - potential efficiency issue
- **West Office** shows solid per-agent productivity
- Consider redistributing resources or investigating East office's success factors

## 📊 Plot 1.4: Revenue vs Win Rate Relationship

In [10]:
# Revenue vs Win Rate scatter
fig = px.scatter(
    team_performance,
    x='win_rate_percent',
    y='total_revenue',
    color='regional_office',
    size='total_opportunities',
    hover_data=['sales_agent', 'won_deals', 'avg_deal_value'],
    title='<b>Revenue vs Win Rate by Sales Agent</b>',
    labels={
        'win_rate_percent': 'Win Rate (%)',
        'total_revenue': 'Total Revenue ($)',
        'regional_office': 'Office'
    },
    color_discrete_map={'East': '#e74c3c', 'Central': '#3498db', 'West': '#2ecc71'}
)

fig.update_layout(
    height=600,
    title_font_size=20
)

fig.show()

### 📝 Key Insights:

- **Positive correlation** between win rate and revenue, but not absolute
- Some high-revenue agents have moderate win rates (high volume strategy)
- **Bubble size** (opportunities handled) shows top performers manage more deals
- East office agents (red) dominate the high-revenue quadrant

---

# ⚠️ SECTION 2: Underperforming Agents Analysis

---

In [11]:
# Filter qualified agents (minimum 10 opportunities)
min_opportunities = 10
qualified_agents = team_performance[team_performance['total_opportunities'] >= min_opportunities].copy()

# Performance categorization
def categorize_performance(win_rate):
    if pd.isna(win_rate):
        return 'No Data'
    elif win_rate < 40:
        return '🔴 Critical'
    elif win_rate < 50:
        return '🟡 Needs Improvement'
    elif win_rate < 60:
        return '🟢 Good'
    else:
        return '⭐ Excellent'

qualified_agents['performance_status'] = qualified_agents['win_rate_percent'].apply(categorize_performance)

print(f"✅ Analyzing {len(qualified_agents)} qualified agents (min {min_opportunities} opportunities)")
print("\nPerformance Status Distribution:")
print(qualified_agents['performance_status'].value_counts())

✅ Analyzing 30 qualified agents (min 10 opportunities)

Performance Status Distribution:
performance_status
⭐ Excellent    26
🟢 Good          4
Name: count, dtype: int64


## 📊 Plot 2.1: Bottom 15 Agents by Win Rate

In [12]:
# Bottom 15 by win rate
bottom_15 = qualified_agents.nsmallest(15, 'win_rate_percent').sort_values('win_rate_percent')

color_map = {
    '🔴 Critical': '#e74c3c',
    '🟡 Needs Improvement': '#f39c12',
    '🟢 Good': '#2ecc71',
    '⭐ Excellent': '#f1c40f'
}

colors = [color_map[status] for status in bottom_15['performance_status']]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=bottom_15['sales_agent'],
    x=bottom_15['win_rate_percent'],
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1)),
    text=[f"{v:.1f}%" for v in bottom_15['win_rate_percent']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Win Rate: %{x:.1f}%<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Bottom 15 Agents by Win Rate</b><br><sub>Color-coded by Performance Status</sub>',
        font=dict(size=20)
    ),
    xaxis_title='Win Rate (%)',
    yaxis_title='Sales Agent',
    height=600,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **No Critical performers** (< 40% win rate) - good baseline performance
- Bottom performers still maintain **55-62% win rates** - acceptable but improvable
- These agents may benefit from:
  - Mentoring from top performers
  - Product knowledge training
  - Lead qualification improvement

## 📊 Plot 2.2: Performance Status Distribution

In [13]:
# Performance status pie chart
status_counts = qualified_agents['performance_status'].value_counts()

fig = go.Figure(data=[go.Pie(
    labels=status_counts.index,
    values=status_counts.values,
    hole=0.4,
    marker=dict(colors=['#f1c40f', '#2ecc71']),
    textinfo='label+percent+value',
    textfont_size=14,
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig.update_layout(
    title=dict(
        text='<b>Performance Status Distribution</b><br><sub>Qualified Agents (≥10 opportunities)</sub>',
        font=dict(size=20)
    ),
    height=500
)

fig.show()

### 📝 Key Insights:

- **86.7%** of agents are in "Excellent" category (≥60% win rate)
- Only **13.3%** in "Good" category (50-60%) - no critical cases
- Overall sales force quality is **very strong**
- Company has effective hiring and training processes

## 📊 Plot 2.3: Average Win Rate by Manager

In [15]:
# Manager performance
manager_stats = qualified_agents.groupby('manager').agg({
    'win_rate_percent': 'mean',
    'sales_agent': 'count',
    'total_revenue': 'sum'
}).rename(columns={'sales_agent': 'agent_count'}).sort_values('win_rate_percent', ascending=False)

overall_avg = qualified_agents['win_rate_percent'].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=manager_stats.index,
    y=manager_stats['win_rate_percent'],
    marker=dict(
        color=manager_stats['win_rate_percent'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title="Win Rate (%)")
    ),
    text=[f"{v:.1f}%<br>({manager_stats.loc[idx, 'agent_count']} agents)" 
          for idx, v in manager_stats['win_rate_percent'].items()],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Avg Win Rate: %{y:.1f}%<extra></extra>'
))

fig.add_hline(
    y=overall_avg,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Overall Avg: {overall_avg:.1f}%",
    annotation_position="top left"
)

fig.update_layout(
    title=dict(
        text='<b>Average Win Rate by Manager</b>',
        font=dict(size=20)
    ),
    xaxis_title='Manager',
    yaxis_title='Average Win Rate (%)',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **Cara Losch** leads with highest team win rate - potential best practices to share
- Most managers cluster around 61-64% - consistent management quality
- **Dustin Brinkmann's** team slightly underperforms - may need support
- Team size varies significantly - larger teams don't guarantee better results

## 📊 Plot 2.4: Opportunities vs Win Rate (Focus on Lower Performers)

In [23]:
# Focus on good and needs improvement categories
focus_agents = qualified_agents[
    qualified_agents['performance_status'].isin(['🟢 Good', '🟡 Needs Improvement'])
]

fig = px.scatter(
    focus_agents,
    x='total_opportunities',
    y='win_rate_percent',
    color='performance_status',
    size='total_revenue',
    hover_data=['sales_agent', 'manager', 'regional_office'],
    title='<b>Lower Performers: Opportunities vs Win Rate</b>',
    labels={
        'total_opportunities': 'Total Opportunities',
        'win_rate_percent': 'Win Rate (%)',
        'performance_status': ''
    },
    color_discrete_map={
        '🟢 Good': '#2ecc71',
        '🟡 Needs Improvement': '#f39c12'
    }
)

fig.add_hline(
    y=60,
    line_dash="dash",
    line_color="gray",

)

fig.update_layout(
    height=600,
    title_font_size=20
)

fig.show()

### 📝 Key Insights:

- Lower performers don't have unusually high/low opportunity counts
- Performance issues are **not volume-related** - focus on quality/training
- Even with 300+ opportunities, some agents stay below 60%
- Consider pairing these agents with high-win-rate mentors

---

# 📈 SECTION 3: Quarter-over-Quarter Trends

---

In [25]:
# Quarterly aggregation
quarterly_trends = sales_pipeline.groupby(['year', 'quarter']).agg({
    'opportunity_id': 'count',
    'close_value': 'sum'
}).rename(columns={'opportunity_id': 'total_opportunities', 'close_value': 'total_revenue'})

# Won/Lost by quarter
won_by_quarter = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby(['year', 'quarter']).size()
lost_by_quarter = sales_pipeline[sales_pipeline['deal_stage'] == 'Lost'].groupby(['year', 'quarter']).size()

quarterly_trends['won_deals'] = won_by_quarter
quarterly_trends['lost_deals'] = lost_by_quarter
quarterly_trends = quarterly_trends.fillna(0)

# Win rate
quarterly_trends['completed_deals'] = quarterly_trends['won_deals'] + quarterly_trends['lost_deals']
quarterly_trends['win_rate_percent'] = (quarterly_trends['won_deals'] / quarterly_trends['completed_deals'] * 100).round(2)

# Average deal value
avg_by_quarter = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby(['year', 'quarter'])['close_value'].mean()
quarterly_trends['avg_deal_value'] = avg_by_quarter.round(0)

# Year-Quarter label
quarterly_trends['year_quarter'] = quarterly_trends.index.get_level_values(0).astype(str) + '-Q' + quarterly_trends.index.get_level_values(1).astype(int).astype(str)

# QoQ changes
quarterly_trends['revenue_change_pct'] = (quarterly_trends['total_revenue'].pct_change() * 100).round(1)
quarterly_trends['opportunities_change_pct'] = (quarterly_trends['total_opportunities'].pct_change() * 100).round(1)

print("✅ Quarterly trends data prepared")
print(f"\nAnalyzing {len(quarterly_trends)} quarters of data")

✅ Quarterly trends data prepared

Analyzing 5 quarters of data


## 📊 Plot 3.1: Total Opportunities by Quarter

In [26]:
# Opportunities trend
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=quarterly_trends['year_quarter'],
    y=quarterly_trends['total_opportunities'],
    mode='lines+markers+text',
    marker=dict(size=12, color='#3498db'),
    line=dict(width=3, color='#3498db'),
    text=[f"{int(v)}" for v in quarterly_trends['total_opportunities']],
    textposition='top center',
    hovertemplate='<b>%{x}</b><br>Opportunities: %{y:,}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Total Opportunities by Quarter</b>',
        font=dict(size=20)
    ),
    xaxis_title='Quarter',
    yaxis_title='Total Opportunities',
    height=500,
    showlegend=False,
    hovermode='x unified'
)

fig.show()

### 📝 Key Insights:

- **Massive spike** in Q1 2017 (+352%) - possible new product launch or market expansion
- Continued growth through Q3 2017 (peak at 2,770 opportunities)
- **Sharp drop** in Q4 2017 (-58%) - seasonal effect or data collection period ending
- Q4 2016 baseline shows company was ramping up operations

## 📊 Plot 3.2: Total Revenue by Quarter

In [27]:
# Revenue trend
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=quarterly_trends['year_quarter'],
    y=quarterly_trends['total_revenue'],
    mode='lines+markers+text',
    marker=dict(size=12, color='#2ecc71'),
    line=dict(width=3, color='#2ecc71'),
    text=[f"${v/1e6:.1f}M" for v in quarterly_trends['total_revenue']],
    textposition='top center',
    fill='tozeroy',
    fillcolor='rgba(46, 204, 113, 0.1)',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Total Revenue by Quarter</b>',
        font=dict(size=20)
    ),
    xaxis_title='Quarter',
    yaxis_title='Total Revenue ($)',
    height=500,
    showlegend=False,
    hovermode='x unified'
)

fig.show()

### 📝 Key Insights:

- Revenue growth mirrors opportunity growth but with variations
- **Peak revenue** in Q2 2017 ($2.97M) despite more opportunities in Q3
- Q2 had higher average deal values than Q3
- **Q4 2017 decline** (-44.5%) more severe than opportunity drop - suggests end of fiscal year timing

## 📊 Plot 3.3: Win Rate Trend by Quarter

In [28]:
# Win rate trend with average line
avg_win_rate = quarterly_trends['win_rate_percent'].mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=quarterly_trends['year_quarter'],
    y=quarterly_trends['win_rate_percent'],
    mode='lines+markers+text',
    marker=dict(size=12, color='#e74c3c'),
    line=dict(width=3, color='#e74c3c'),
    text=[f"{v:.1f}%" for v in quarterly_trends['win_rate_percent']],
    textposition='top center',
    hovertemplate='<b>%{x}</b><br>Win Rate: %{y:.1f}%<extra></extra>'
))

fig.add_hline(
    y=avg_win_rate,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Avg: {avg_win_rate:.1f}%",
    annotation_position="right"
)

fig.update_layout(
    title=dict(
        text='<b>Win Rate Trend by Quarter</b>',
        font=dict(size=20)
    ),
    xaxis_title='Quarter',
    yaxis_title='Win Rate (%)',
    height=500,
    showlegend=False,
    hovermode='x unified'
)

fig.show()

### 📝 Key Insights:

- **Declining trend** in win rate from 82% (Q4 2016) to 60.7% (Q4 2017)
- Possible causes:
  - Rapid expansion led to less qualified leads
  - New sales agents still learning (lower individual win rates)
  - Increased competition in market
- Stabilization around 60-62% in latter half of 2017
- **Action needed**: Focus on lead quality and agent training

## 📊 Plot 3.4: Quarter-over-Quarter Revenue Growth Rate

In [29]:
# QoQ growth rate
qoq_data = quarterly_trends[1:]  # Skip first quarter (no previous comparison)

colors = ['green' if x > 0 else 'red' for x in qoq_data['revenue_change_pct']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=qoq_data['year_quarter'],
    y=qoq_data['revenue_change_pct'],
    marker=dict(color=colors, line=dict(color='white', width=1)),
    text=[f"{v:+.1f}%" for v in qoq_data['revenue_change_pct']],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>QoQ Change: %{y:.1f}%<extra></extra>'
))

fig.add_hline(
    y=0,
    line_color="black",
    line_width=2
)

fig.update_layout(
    title=dict(
        text='<b>Quarter-over-Quarter Revenue Growth Rate</b>',
        font=dict(size=20)
    ),
    xaxis_title='Quarter',
    yaxis_title='Revenue Change (%)',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **Explosive Q1 2017 growth** (+314.6%) indicates major business event
- Continued positive growth through Q2 (+36.8%)
- **First decline** in Q3 2017 (-5.9%) - early warning sign
- **Severe Q4 drop** (-44.5%) requires investigation:
  - Holiday season slowdown?
  - Budget exhaustion at client companies?
  - End of data collection period?
- Need to develop Q4 revenue strategies for future years

## 📊 Plot 3.5: Combined Quarterly Metrics Dashboard

In [30]:
# Multi-metric quarterly view
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Opportunities', 'Revenue ($)', 
        'Win Rate (%)', 'Avg Deal Value ($)'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Opportunities
fig.add_trace(
    go.Scatter(
        x=quarterly_trends['year_quarter'],
        y=quarterly_trends['total_opportunities'],
        mode='lines+markers',
        marker=dict(size=8, color='#3498db'),
        line=dict(width=2),
        name='Opportunities'
    ),
    row=1, col=1
)

# Revenue
fig.add_trace(
    go.Scatter(
        x=quarterly_trends['year_quarter'],
        y=quarterly_trends['total_revenue'],
        mode='lines+markers',
        marker=dict(size=8, color='#2ecc71'),
        line=dict(width=2),
        fill='tozeroy',
        name='Revenue'
    ),
    row=1, col=2
)

# Win Rate
fig.add_trace(
    go.Scatter(
        x=quarterly_trends['year_quarter'],
        y=quarterly_trends['win_rate_percent'],
        mode='lines+markers',
        marker=dict(size=8, color='#e74c3c'),
        line=dict(width=2),
        name='Win Rate'
    ),
    row=2, col=1
)

# Avg Deal Value
fig.add_trace(
    go.Scatter(
        x=quarterly_trends['year_quarter'],
        y=quarterly_trends['avg_deal_value'],
        mode='lines+markers',
        marker=dict(size=8, color='#9b59b6'),
        line=dict(width=2),
        name='Avg Deal Value'
    ),
    row=2, col=2
)

fig.update_layout(
    title=dict(
        text='<b>Quarterly Performance Dashboard</b>',
        font=dict(size=20)
    ),
    height=700,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- All four metrics show similar pattern: spike in early 2017, stabilization, then Q4 drop
- **Average deal value** remained relatively stable ($2,100-$2,400) despite volume changes
- Company maintained deal quality even during rapid expansion
- The synchronized drop across all metrics in Q4 suggests systematic/seasonal factor

---

# 🏅 SECTION 4: Product Win Rate Analysis

---

In [31]:
# Product performance analysis
product_performance = sales_pipeline.groupby('product').agg({
    'opportunity_id': 'count',
    'close_value': 'sum'
}).rename(columns={'opportunity_id': 'total_opportunities', 'close_value': 'total_revenue'})

# Won/Lost by product
won_by_product = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby('product').size()
lost_by_product = sales_pipeline[sales_pipeline['deal_stage'] == 'Lost'].groupby('product').size()

product_performance['won_deals'] = won_by_product
product_performance['lost_deals'] = lost_by_product
product_performance = product_performance.fillna(0)

# Win rate
product_performance['completed_deals'] = product_performance['won_deals'] + product_performance['lost_deals']
product_performance['win_rate_percent'] = (product_performance['won_deals'] / product_performance['completed_deals'] * 100).round(2)

# Merge with product details
product_performance = product_performance.merge(
    products[['product', 'series', 'sales_price']], 
    left_index=True, 
    right_on='product',
    how='left'
)

# Average actual sale price
avg_sale_by_product = sales_pipeline[sales_pipeline['deal_stage'] == 'Won'].groupby('product')['close_value'].mean()
product_performance['avg_sale_price'] = avg_sale_by_product.round(0)

# Discount
product_performance['avg_discount_percent'] = (
    (1 - product_performance['avg_sale_price'] / product_performance['sales_price']) * 100
).round(2)

# Performance tier
def product_tier(win_rate):
    if pd.isna(win_rate):
        return 'No Data'
    elif win_rate >= 60:
        return '⭐⭐⭐ Excellent'
    elif win_rate >= 50:
        return '⭐⭐ Good'
    elif win_rate >= 40:
        return '⭐ Fair'
    else:
        return '❌ Poor'

product_performance['performance_tier'] = product_performance['win_rate_percent'].apply(product_tier)

print(f"✅ Analyzing {len(product_performance)} products")
print("\nProduct Performance Tiers:")
print(product_performance['performance_tier'].value_counts())

✅ Analyzing 7 products

Product Performance Tiers:
performance_tier
⭐⭐⭐ Excellent    7
Name: count, dtype: int64


## 📊 Plot 4.1: Win Rate by Product

In [32]:
# Win rate by product
product_sorted = product_performance.sort_values('win_rate_percent', ascending=True)

tier_colors = {
    '⭐⭐⭐ Excellent': '#f1c40f',
    '⭐⭐ Good': '#2ecc71',
    '⭐ Fair': '#f39c12',
    '❌ Poor': '#e74c3c'
}

colors = [tier_colors[tier] for tier in product_sorted['performance_tier']]
avg_product_win_rate = product_performance['win_rate_percent'].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    y=product_sorted['product'],
    x=product_sorted['win_rate_percent'],
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1)),
    text=[f"{v:.1f}%" for v in product_sorted['win_rate_percent']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Win Rate: %{x:.1f}%<extra></extra>'
))

fig.add_vline(
    x=avg_product_win_rate,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Avg: {avg_product_win_rate:.1f}%",
    annotation_position="top"
)

fig.update_layout(
    title=dict(
        text='<b>Win Rate by Product</b><br><sub>Color-coded by Performance Tier</sub>',
        font=dict(size=20)
    ),
    xaxis_title='Win Rate (%)',
    yaxis_title='Product',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **All products** perform excellently (60-65% win rate) - strong product-market fit
- **MG Special** leads with 64.8% win rate
- **GTK 500** has lowest win rate (60%) but still "Excellent" tier
- Very narrow range (60-65%) suggests consistent product quality
- Win rates cluster tightly around 62.7% average

## 📊 Plot 4.2: Total Revenue by Product

In [33]:
# Revenue by product
product_revenue_sorted = product_performance.sort_values('total_revenue', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=product_revenue_sorted['product'],
    y=product_revenue_sorted['total_revenue'],
    marker=dict(
        color=product_revenue_sorted['total_revenue'],
        colorscale='Greens',
        showscale=True,
        colorbar=dict(title="Revenue ($)")
    ),
    text=[f"${v/1e6:.1f}M" for v in product_revenue_sorted['total_revenue']],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>Total Revenue by Product</b>',
        font=dict(size=20)
    ),
    xaxis_title='Product',
    yaxis_title='Total Revenue ($)',
    height=500,
    showlegend=False
)

fig.show()

### 📝 Key Insights:

- **GTX Pro** dominates revenue ($3.51M) despite not having highest win rate
- High-priced products (GTX Pro, GTX Plus Pro) drive majority of revenue
- **GTX series** collectively generates ~75% of total revenue
- **GTK 500** has lowest revenue due to low volume (40 opportunities)
- Revenue distribution is more uneven than win rate distribution

## 📊 Plot 4.3: Revenue vs Win Rate by Product

In [34]:
# Revenue vs Win Rate with bubble size = opportunities
fig = px.scatter(
    product_performance,
    x='win_rate_percent',
    y='total_revenue',
    size='total_opportunities',
    color='series',
    text='product',
    title='<b>Product Performance: Revenue vs Win Rate</b><br><sub>Bubble size = Total Opportunities</sub>',
    labels={
        'win_rate_percent': 'Win Rate (%)',
        'total_revenue': 'Total Revenue ($)',
        'series': 'Product Series'
    },
    color_discrete_map={'GTX': '#3498db', 'MG': '#e74c3c', 'GTK': '#f39c12'}
)

fig.update_traces(textposition='top center')

fig.update_layout(
    height=600,
    title_font_size=20
)

fig.show()

### 📝 Key Insights:

- **GTX series** (blue) dominates both revenue and volume
- **MG series** (red) has high win rates but lower revenue per product
- **GTK 500** (orange) is an outlier - low volume, high price point
- No strong correlation between win rate and revenue (price/volume matters more)
- Largest bubbles (GTX Basic, MG Special) show high-volume products

## 📊 Plot 4.4: Product Series Comparison

In [35]:
# Series-level aggregation
series_performance = product_performance.groupby('series').agg({
    'total_opportunities': 'sum',
    'won_deals': 'sum',
    'lost_deals': 'sum',
    'total_revenue': 'sum',
    'product': 'count'
}).rename(columns={'product': 'product_count'})

series_performance['win_rate_percent'] = (
    series_performance['won_deals'] / 
    (series_performance['won_deals'] + series_performance['lost_deals']) * 100
).round(2)

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Total Revenue', 'Total Opportunities', 'Avg Win Rate'),
    specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "bar"}]]
)

colors_series = {'GTX': '#3498db', 'MG': '#e74c3c', 'GTK': '#f39c12'}

# Revenue
for series in series_performance.index:
    fig.add_trace(
        go.Bar(
            x=[series],
            y=[series_performance.loc[series, 'total_revenue']],
            marker_color=colors_series[series],
            showlegend=False,
            text=f"${series_performance.loc[series, 'total_revenue']/1e6:.1f}M",
            textposition='outside'
        ),
        row=1, col=1
    )

# Opportunities
for series in series_performance.index:
    fig.add_trace(
        go.Bar(
            x=[series],
            y=[series_performance.loc[series, 'total_opportunities']],
            marker_color=colors_series[series],
            showlegend=False,
            text=f"{int(series_performance.loc[series, 'total_opportunities'])}",
            textposition='outside'
        ),
        row=1, col=2
    )

# Win Rate
for series in series_performance.index:
    fig.add_trace(
        go.Bar(
            x=[series],
            y=[series_performance.loc[series, 'win_rate_percent']],
            marker_color=colors_series[series],
            showlegend=False,
            text=f"{series_performance.loc[series, 'win_rate_percent']:.1f}%",
            textposition='outside'
        ),
        row=1, col=3
    )

fig.update_layout(
    title=dict(
        text='<b>Product Series Performance Comparison</b>',
        font=dict(size=20)
    ),
    height=500
)

fig.show()

### 📝 Key Insights:

- **GTX series**: Highest revenue ($7.34M), highest volume (5,697 opps), competitive win rate (63.3%)
- **MG series**: Good revenue ($2.26M), high volume (3,063 opps), **best win rate (62.3%)**
- **GTK series**: Lowest metrics across the board (niche product)
- **GTX is the cash cow** - focus marketing and sales training here
- MG series has potential for revenue growth given high win rate

## 📊 Plot 4.5: Product Opportunity Distribution

In [36]:
# Treemap of product opportunities
fig = px.treemap(
    product_performance,
    path=['series', 'product'],
    values='total_opportunities',
    color='win_rate_percent',
    color_continuous_scale='RdYlGn',
    title='<b>Product Portfolio - Opportunity Distribution</b><br><sub>Size = Total Opportunities | Color = Win Rate</sub>',
    hover_data={'total_revenue': ':$,.0f', 'won_deals': True}
)

fig.update_traces(
    textposition='middle center',
    textfont_size=12
)

fig.update_layout(
    height=600,
    title_font_size=20
)

fig.show()

### 📝 Key Insights:

- **GTX Basic** is the largest single product by volume (21% of all opportunities)
- **MG Special** is second largest (19%)
- Product portfolio is relatively **balanced** - no over-dependence on single product
- Green shading shows all products have healthy win rates (60%+)
- GTK 500's tiny size confirms it's a specialized niche offering

---

# 📋 EXECUTIVE SUMMARY & RECOMMENDATIONS

---

In [37]:
# Calculate key metrics
total_opps = len(sales_pipeline)
won_deals_total = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Won'])
overall_win_rate = (won_deals_total / len(sales_pipeline[sales_pipeline['deal_stage'].isin(['Won', 'Lost'])]) * 100)
total_revenue_won = sales_pipeline[sales_pipeline['deal_stage'] == 'Won']['close_value'].sum()
avg_deal_size = total_revenue_won / won_deals_total

print("="*80)
print("EXECUTIVE SUMMARY - B2B SALES PIPELINE ANALYSIS")
print("="*80)

print("\n📊 OVERALL PERFORMANCE:")
print(f"   • Total Opportunities: {total_opps:,}")
print(f"   • Won Deals: {won_deals_total:,}")
print(f"   • Overall Win Rate: {overall_win_rate:.2f}%")
print(f"   • Total Revenue: ${total_revenue_won:,.0f}")
print(f"   • Average Deal Size: ${avg_deal_size:,.0f}")

print("\n🏆 TOP PERFORMERS:")
top_agent = team_performance.nlargest(1, 'total_revenue').iloc[0]
print(f"   • Best Agent: {top_agent['sales_agent']} (${top_agent['total_revenue']:,.0f} revenue)")
best_product = product_performance.nlargest(1, 'win_rate_percent').iloc[0]
print(f"   • Best Product: {best_product['product']} ({best_product['win_rate_percent']:.1f}% win rate)")
top_office = team_performance.groupby('regional_office')['total_revenue'].sum().idxmax()
print(f"   • Top Regional Office: {top_office}")

print("\n⚠️ AREAS OF CONCERN:")
if len(qualified_agents) > 0:
    worst_agent = qualified_agents.nsmallest(1, 'win_rate_percent').iloc[0]
    print(f"   • Lowest Win Rate Agent: {worst_agent['sales_agent']} ({worst_agent['win_rate_percent']:.1f}%)")
worst_product = product_performance.nsmallest(1, 'win_rate_percent').iloc[0]
print(f"   • Lowest Win Rate Product: {worst_product['product']} ({worst_product['win_rate_percent']:.1f}%)")
print(f"   • Win Rate Decline: From 82% (Q4 2016) to 61% (Q4 2017)")
print(f"   • Q4 2017 Revenue Drop: -44.5% QoQ")

print("\n💡 KEY RECOMMENDATIONS:")
print("\n1. ADDRESS WIN RATE DECLINE:")
print("   • Implement lead qualification scoring system")
print("   • Enhance onboarding training for new agents")
print("   • Analyze Q4 2016 practices (82% win rate) for replication")

print("\n2. OPTIMIZE Q4 PERFORMANCE:")
print("   • Develop Q4-specific sales strategies")
print("   • Increase marketing spend in Q3 to build Q4 pipeline")
print("   • Offer Q4 incentives to counteract seasonal slowdown")

print("\n3. LEVERAGE TOP PERFORMERS:")
print("   • Create mentorship program pairing top/bottom performers")
print("   • Document and share Cara Losch's team management practices")
print("   • Study East office success factors for company-wide rollout")

print("\n4. PRODUCT PORTFOLIO OPTIMIZATION:")
print("   • Double down on GTX series marketing (75% of revenue)")
print("   • Investigate MG series revenue growth potential (high win rate)")
print("   • Re-evaluate GTK 500 market positioning (low volume)")

print("\n5. REGIONAL STRATEGY:")
print("   • Replicate East office tactics in Central region")
print("   • Balance agent distribution based on market potential")
print("   • Consider expansion in high-performing territories")

print("\n" + "="*80)

EXECUTIVE SUMMARY - B2B SALES PIPELINE ANALYSIS

📊 OVERALL PERFORMANCE:
   • Total Opportunities: 8,800
   • Won Deals: 4,238
   • Overall Win Rate: 63.15%
   • Total Revenue: $10,005,534
   • Average Deal Size: $2,361

🏆 TOP PERFORMERS:
   • Best Agent: Darcel Schlecht ($1,153,214 revenue)
   • Best Product: MG Special (64.8% win rate)
   • Top Regional Office: West

⚠️ AREAS OF CONCERN:
   • Lowest Win Rate Agent: Lajuana Vencill (55.0%)
   • Lowest Win Rate Product: GTK 500 (60.0%)
   • Win Rate Decline: From 82% (Q4 2016) to 61% (Q4 2017)
   • Q4 2017 Revenue Drop: -44.5% QoQ

💡 KEY RECOMMENDATIONS:

1. ADDRESS WIN RATE DECLINE:
   • Implement lead qualification scoring system
   • Enhance onboarding training for new agents
   • Analyze Q4 2016 practices (82% win rate) for replication

2. OPTIMIZE Q4 PERFORMANCE:
   • Develop Q4-specific sales strategies
   • Increase marketing spend in Q3 to build Q4 pipeline
   • Offer Q4 incentives to counteract seasonal slowdown

3. LEVERAGE TO

---

## 🎯 Conclusion

This analysis reveals a **generally healthy sales organization** with strong win rates (63%) and solid revenue generation ($10M+). However, several trends require attention:

### ✅ Strengths:
- High overall win rate (63.2%) with 86.7% of agents in "Excellent" category
- Strong product-market fit across entire portfolio (all products 60%+ win rate)
- Successful business expansion in 2017 (352% opportunity growth in Q1)
- Balanced revenue distribution across product lines

### ⚠️ Challenges:
- **Win rate erosion**: 21% decline from Q4 2016 to Q4 2017
- **Q4 revenue cliff**: 44.5% drop requires strategic intervention
- Performance concentration in small group of top agents
- Potential lead quality issues due to rapid expansion

### 🚀 Next Steps:
1. Implement recommended actions from Executive Summary
2. Monitor win rate trends monthly (target: return to 70%+)
3. Develop Q4 contingency plan for 2018
4. Establish knowledge transfer programs from top performers
5. Re-run this analysis quarterly to track improvements

---

**Analysis Date**: April 2026  
**Data Period**: October 2016 - December 2017  
**Total Records Analyzed**: 8,800 opportunities

---